# 🌍 Global DVM / Trendlyne Screener — Colab Demo

A zero-install, self-contained tour of the **Global Market Scanners** platform. It fetches
live prices + fundamentals via `yfinance` and reproduces the core scoring:

- **Momentum (0-100)** from RSI, MFI, MACD, moving-average stack, 52w-high proximity
- **Durability (0-100)** from ROE, D/E, growth, margins
- **Valuation (0-100)** from earnings-yield / P/B rank
- **DVM composite** → Trendlyne `GGG` / `GGB` / `BBG` classification

> The full platform runs these across **19 markets** on local data with a Cassandra/Kafka/Flink
> backbone. This notebook is a standalone demo on a small live universe. See the
> [User Guide](https://github.com/herrrickshaw/global-market-scanners/blob/main/USER_GUIDE.md).

In [ ]:
!pip -q install yfinance pandas numpy scikit-learn

## 1. Pick a universe & fetch prices
A small cross-market set of liquid names (yfinance symbols; add your own).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, yfinance as yf

UNIVERSE = ['AAPL','MSFT','NVDA','JPM','BAC','XOM','JNJ','WMT','KO','PG',      # US
            'SAP.DE','RHM.DE','SIE.DE','ASML.AS','MC.PA',                       # Europe
            '7203.T','6758.T','8306.T','9984.T',                               # Japan
            'NWG.L','LLOY.L','BARC.L','STAN.L','BATS.L',                        # UK
            'CM.TO','MFC.TO','QBE.AX']                                         # Canada/Australia

data = yf.download(UNIVERSE, period='2y', auto_adjust=True, group_by='ticker', progress=False)
ohlc = {t: data[t].dropna(how='all') for t in UNIVERSE if t in data.columns.get_level_values(0)}
print(f'fetched {len(ohlc)} tickers')

## 2. Momentum score + Trendlyne technical metrics
RSI, MFI, MACD, DMA-stack, 52w-high proximity → a 0-100 Momentum composite, plus the
`trendlyne_technical` screen (RSI 50-70 & MFI>=50 & MACD bullish).

In [ ]:
def technicals(df):
    c=df['Close'].astype(float); h=df['High'].astype(float); low=df['Low'].astype(float); v=df['Volume'].astype(float)
    if len(c)<200: return None
    d=c.diff()
    rsi=(100-100/(1+d.clip(lower=0).rolling(14).mean()/(-d.clip(upper=0)).rolling(14).mean().replace(0,np.nan))).iloc[-1]
    mh=(c.ewm(span=12).mean()-c.ewm(span=26).mean()); mh=(mh-mh.ewm(span=9).mean()).iloc[-1]
    dma50=c.rolling(50).mean().iloc[-1]; dma200=c.rolling(200).mean().iloc[-1]; px=c.iloc[-1]
    hi52=c.rolling(252,min_periods=150).max().iloc[-1]
    tp=(h+low+c)/3; rmf=tp*v
    pos=rmf.where(tp.diff()>0,0).rolling(14).sum(); neg=rmf.where(tp.diff()<0,0).rolling(14).sum().replace(0,np.nan)
    mfi=(100-100/(1+pos/neg)).iloc[-1]
    subs=[min(100,max(0, rsi if rsi<=70 else 70-(rsi-70)*2)),
          100 if mh>0 else 25,
          100 if px>dma50>dma200 else (60 if px>dma200 else 20),
          min(100,max(0,100+(px/hi52-1)*300))]
    return dict(M=round(float(np.mean(subs)),1), rsi=round(float(rsi),1), mfi=round(float(mfi),1),
                macd_bull=bool(mh>0), above_200=bool(px>dma200))

tech = {t: technicals(df) for t,df in ohlc.items()}
tech = {t:v for t,v in tech.items() if v}
tdf = pd.DataFrame(tech).T
tdf['trendlyne_technical'] = tdf.apply(lambda r: 50<=r['rsi']<=70 and r['mfi']>=50 and r['macd_bull'], axis=1)
print('trendlyne_technical hits:'); tdf[tdf['trendlyne_technical']][['M','rsi','mfi']]

## 3. Fundamentals → Durability + Valuation → DVM composite
Pull ROE / D-E / P/E / P/B via `yfinance`, score Durability & Valuation, and classify each
stock **GGG / GGB / BBG** (Durability, Valuation, Momentum each Good≥50 / Bad<50).

In [ ]:
LABELS={'GGG':'Strong Performer','GGB':'Value Under Radar','GBG':'Expensive Durable Mover',
        'GBB':'Expensive Quality','BGG':'Cheap Turnaround Mover','BGB':'Deep Value / Watch',
        'BBG':'Momentum Trap','BBB':'Weak / Avoid'}

def norm_de(x): return None if x is None else (x/100 if x>10 else x)
rows=[]
for t in tdf.index:
    try: i=yf.Ticker(t).get_info()
    except Exception: i={}
    roe=(i.get('returnOnEquity') or 0)*100; de=norm_de(i.get('debtToEquity'))
    rg=(i.get('revenueGrowth') or 0)*100; om=(i.get('operatingMargins') or 0)*100
    pe=i.get('trailingPE'); pb=i.get('priceToBook')
    dsub=[90 if roe>20 else 75 if roe>15 else 55 if roe>10 else 35 if roe>0 else 15,
          90 if (de is not None and de<0.5) else 70 if (de is not None and de<1) else 45 if (de is not None and de<2) else 20,
          85 if rg>15 else 60 if rg>5 else 40 if rg>0 else 20,
          85 if om>20 else 60 if om>10 else 40 if om>0 else 20]
    rows.append(dict(ticker=t, M=tdf.loc[t,'M'], D=round(float(np.mean(dsub)),1),
                     ey=(1/pe if (pe and pe>0) else np.nan), pb=pb,
                     roe=round(roe,1), de=de, pe=pe))
df=pd.DataFrame(rows)
df['V']=(0.6*df['ey'].rank(pct=True)*100 + 0.4*(1-df['pb'].rank(pct=True))*100).round(1)
g=lambda x: 'G' if x>=50 else 'B'
df['code']=[g(d)+g(v)+g(m) for d,v,m in zip(df['D'],df['V'],df['M'])]
df['label']=df['code'].map(LABELS)
df['composite']=((df['D']+df['V']+df['M'])/3).round(1)
df.sort_values('composite',ascending=False)[['ticker','D','V','M','composite','code','label']].head(15)

## 4. The GGG 'Strong Performers'
High on all three axes — durable, reasonably valued, and with momentum.

In [ ]:
df[df['code']=='GGG'].sort_values('composite',ascending=False)[['ticker','D','V','M','composite','roe','de','pe']]

## Next steps
This demo runs on a handful of live tickers. The full platform does all of this across
**19 markets** on cached local data, plus point-in-time backtesting, factor research, ML
screen discovery, and a Cassandra/Kafka/Flink backbone.

- **[User Guide](https://github.com/herrrickshaw/global-market-scanners/blob/main/USER_GUIDE.md)** — every workflow
- **[DVM Composite](https://github.com/herrrickshaw/global-market-scanners/blob/main/DVM_COMPOSITE.md)** — global GGG results
- **[Architecture](https://github.com/herrrickshaw/global-market-scanners/blob/main/ARCHITECTURE.md)** — the data backbone

```bash
git clone https://github.com/herrrickshaw/global-market-scanners
pip install -r requirements.txt
python dvm_global.py --screen trendlyne_technical   # all 19 markets
```